In [ ]:
#!pip install peft optuna scikit-learn tensorboard

In [3]:
import numpy as np
import evaluate
import torch
from datasets import load_dataset
import csv
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    EarlyStoppingCallback # Pour arrêter si ça ne s'améliore plus (Monitoring/Efficiency)
)
from peft import LoraConfig, get_peft_model, TaskType # Pour l'efficacité (LoRA)

# --- Configuration ---
MODEL_CHECKPOINT = "bert-base-uncased"
NUM_LABELS = 6
BATCH_SIZE = 16

/home/onyxia/work/venv_sa/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Chargement (Votre code original fonctionnait très bien ici)
col_names = [
    "id", "label_text", "statement", "subject", "speaker", 
    "job_title", "state_info", "party_affiliation", 
    "barely_true_counts", "false_counts", "half_true_counts", 
    "mostly_true_counts", "pants_on_fire_counts", "context"
]

raw_datasets = load_dataset(
    "csv", 
    data_files={
        "train": "/home/onyxia/work/Stat_App/Data/train.tsv", 
        "validation": "/home/onyxia/work/Stat_App/Data/valid.tsv", 
        "test": "/home/onyxia/work/Stat_App/Data/test.tsv"
    }, 
    delimiter="\t", column_names=col_names, quoting=csv.QUOTE_NONE
)

# Mapping Labels
label_mapping = {'pants-fire': 0, 'false': 1, 'barely-true': 2, 'half-true': 3, 'mostly-true': 4, 'true': 5}
raw_datasets = raw_datasets.map(lambda x: {'label': label_mapping[x['label_text']]})

# Tokenization
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
def tokenize_function(example):
    return tokenizer(example["statement"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Map: 100%|██████████| 1283/1283 [00:00<00:00, 8454.68 examples/s]


In [5]:
# Chargement des métriques
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    
    # Calcul de toutes les métriques
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    
    # 'weighted' prend en compte le déséquilibre des classes
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")
    precision = precision_metric.compute(predictions=predictions, references=labels, average="weighted")
    recall = recall_metric.compute(predictions=predictions, references=labels, average="weighted")
    
    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"],
        "precision": precision["precision"],
        "recall": recall["recall"]
    }

In [6]:
# Fonction d'initialisation du modèle (nécessaire pour Optuna plus tard)
def model_init():
    # 1. Charger le modèle de base
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT, 
        num_labels=NUM_LABELS
    )
    
    # 2. Configuration LoRA (Parameter-Efficient Fine-Tuning)
    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS, 
        inference_mode=False, 
        r=8,            # Rang de la matrice (plus petit = plus léger)
        lora_alpha=16, 
        lora_dropout=0.1
    )
    
    # 3. Appliquer LoRA au modèle
    model = get_peft_model(model, peft_config)
    
    # Affiche le % de paramètres entraînables (souvent < 1%)
    model.print_trainable_parameters()
    
    return model

In [ ]:
training_args = TrainingArguments(
    output_dir="liar-bert-lora-finetuned",
    
    # --- Efficiency ---
    gradient_checkpointing=True,  # Économise beaucoup de VRAM
    fp16=True,                    # Utilise la précision mixte (plus rapide sur GPU)
    
    # --- Monitoring ---
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,             # Log plus fréquent pour voir la courbe
    report_to="tensorboard",      # Active le suivi
    load_best_model_at_end=True,
    metric_for_best_model="f1",   # On optimise le F1, pas juste l'accuracy
    
    # --- Standard ---
    learning_rate=2e-4,           # LoRA supporte des taux plus élevés que le fine-tuning classique
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=3,           # On peut augmenter car c'est plus rapide
    weight_decay=0.01,
    save_total_limit=1            # Économie d'espace disque
)

In [10]:
trainer = Trainer(
    model_init=model_init,        # On passe la fonction, pas le modèle instancié
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # Arrête si pas d'amélioration après 2 époques
)

def hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [16, 32]),
    }

best_run = trainer.hyperparameter_search(
    direction="maximize", 
    backend="optuna", 
    hp_space=hp_space, 
    n_trials=5  # Tester 5 combinaisons différentes
)
print(f"Meilleurs paramètres trouvés : {best_run.hyperparameters}")

# Appliquer les meilleurs paramètres pour l'entraînement final
for n, v in best_run.hyperparameters.items():
    setattr(trainer.args, n, v)


# Entraînement Final
trainer.train()

/tmp/ipykernel_12751/3203344984.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 299,526 || all params: 109,786,380 || trainable%: 0.2728


[I 2025-12-07 11:15:22,714] A new study created in memory with name: no-name-95fa8a76-06cf-43af-ba14-923060dd8321
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 299,526 || all params: 109,786,380 || trainable%: 0.2728


/home/onyxia/work/venv_sa/lib/python3.13/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


...

Meilleurs paramètres trouvés : {'learning_rate': 0.00011711565349759347, 'per_device_train_batch_size': 32}
trainable params: 299,526 || all params: 109,786,380 || trainable%: 0.2728


/home/onyxia/work/venv_sa/lib/python3.13/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.774700,1.758718,0.204829,0.111385,0.253069,0.204829
2,1.753900,1.765766,0.204050,0.115731,0.145468,0.204050
3,1.749700,1.758793,0.211059,0.117219,0.164907,0.211059


/home/onyxia/work/venv_sa/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/onyxia/work/venv_sa/lib/python3.13/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/home/onyxia/work/venv_sa/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/onyxia/work/venv_sa/lib/python3.13/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Grad

TrainOutput(global_step=963, training_loss=1.7618985735614971, metrics={'train_runtime': 75.0779, 'train_samples_per_second': 410.334, 'train_steps_per_second': 12.827, 'total_flos': 2033575066156032.0, 'train_loss': 1.7618985735614971, 'epoch': 3.0})

In [11]:
print("--- Évaluation Complète sur Test ---")
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)

--- Évaluation Complète sur Test ---


{'eval_loss': 1.7422270774841309, 'eval_accuracy': 0.21590023382696805, 'eval_f1': 0.11850862071317726, 'eval_precision': 0.12082558772507987, 'eval_recall': 0.21590023382696805, 'eval_runtime': 2.6743, 'eval_samples_per_second': 479.758, 'eval_steps_per_second': 30.289, 'epoch': 3.0}


/home/onyxia/work/venv_sa/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
